# 3. Model configuration and training

Model construction is staged: 
- select a model family and name, 
- configure components or apply a preset, 
- configure a dataset, 
- compile, 
- estimate resources, 
- train. 

Compilation creates the PyTorch module and dataset but marks the model untrained until `fit` succeeds.

In [3]:
import os 
from pathlib import Path

# seting global dir
cwd=Path.cwd()
if cwd.name == "tutorials":
    # os.chdir(cwd.parent.parent) 
    os.chdir(cwd.parent.parent.parent) 
os.getcwd()

'/home/maxi7524/repositories/MSIAutoEncoderWrapper'

In [5]:
from pathlib import Path
from msi_autoencoder_wrapper.core.wrapper import MSIAutoEncoderWrapper

wrapper = MSIAutoEncoderWrapper("tutorial_workspace")
# more optimal is code below, but here we want to ensure user provided right structure
# wrapper.workspace.set_default_image_path('example')
image_path = Path("data/tutorial_workspace/imgs/example.imzML").resolve()
wrapper.context_manager.set_reader("PyImzMLReader", str(image_path))
wrapper.context_manager.set_binner("LinearBinning", str(image_path), bin_step=0.1)
wrapper.context_manager.set_inverse_binner(
    "TopPeaksInverseBinner", str(image_path), max_bins=500, window_size=3
)
wrapper.workspace.set_active_image(str(image_path))

2026-07-19 13:32:50,430 | INFO     | msi_autoencoder_wrapper.core.wrapper:70 | MSIAutoEncoderWrapper: Anchoring processing device state: cuda
2026-07-19 13:32:50,432 | INFO     | msi_autoencoder_wrapper.core.mixins.context_manager.context_manager_mixin:46 | Enforcing automatic module discovery for reader and binner registries.
2026-07-19 13:32:50,433 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 2 implementation module(s) in package 'msi_autoencoder_wrapper.readers.strategies'.
2026-07-19 13:32:50,434 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 1 implementation module(s) in package 'msi_autoencoder_wrapper.binners.binners_strategies'.
2026-07-19 13:32:50,436 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 1 implementation module(s) in package 'msi_autoencoder_wrapper.binners.inverse_strategies'.
2026-07-19 13:32:50,437 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 9 implementatio

/home/maxi7524/micromamba/envs/ims_env/lib/python3.12/site-packages/pyimzml/ontology/ontology.py:92: UserWarning: Accession MS:1000563 found with incorrect name "Thermo RAW file". Updating name to "Thermo RAW format".
  warn(
/home/maxi7524/micromamba/envs/ims_env/lib/python3.12/site-packages/pyimzml/ontology/ontology.py:92: UserWarning: Accession MS:1000590 found with incorrect name "contact organization". Updating name to "contact affiliation".
  warn(
/home/maxi7524/micromamba/envs/ims_env/lib/python3.12/site-packages/pyimzml/ontology/ontology.py:92: UserWarning: Accession IMS:1000042 found with incorrect name "max count of pixel x". Updating name to "max count of pixels x".
  warn(
/home/maxi7524/micromamba/envs/ims_env/lib/python3.12/site-packages/pyimzml/ontology/ontology.py:92: UserWarning: Accession IMS:1000043 found with incorrect name "max count of pixel y". Updating name to "max count of pixels y".
  warn(


2026-07-19 13:32:52,924 | INFO     | msi_autoencoder_wrapper.core.mixins.context_manager.context_manager_mixin:338 | Successfully registered component 'reader' into ledger for image 'example'
2026-07-19 13:32:52,926 | INFO     | msi_autoencoder_wrapper.core.mixins.workspace.proxies.getters_and_setters_proxy:69 | Active image set via direct filesystem path: example (Location: /home/maxi7524/repositories/MSIAutoEncoderWrapper/data/tutorial_workspace/imgs)
2026-07-19 13:32:52,927 | INFO     | msi_autoencoder_wrapper.core.mixins.context_manager.context_manager_mixin:323 | Resolving system component 'binner' under image context 'example'
2026-07-19 13:32:52,927 | INFO     | msi_autoencoder_wrapper.core.mixins.active_context.active_context_mixin:106 | Successfully bound active context memory maps for: example
2026-07-19 13:32:52,929 | INFO     | msi_autoencoder_wrapper.core.mixins.context_manager.context_manager_mixin:338 | Successfully registered component 'binner' into ledger for image 'ex

## Inspect the registries

A family determines the valid component categories, criteria, and runtime interface. Discovery is useful before writing a config and returns constructor metadata when `return_value=True`.

In [7]:
# Here we search through all available models and set autoencoder
wrapper.models_manager.get_available_model_types()
wrapper.models_manager.set_model_type("autoencoder", "tutorial-autoencoder")


 Available Master Model Topologies

[Model Type]: 'autoencoder'
 Description: Symmetric architectural backbone coordinating data transformations across autoencoder blocks.
 Parameters (kwargs):
   - resolved_components: Required



In [ ]:
# Here we search through available components for given model (autoencoder) 
wrapper.models_manager.get_available_component_categories()
wrapper.models_manager.get_available_model_presets()
wrapper.models_manager.get_available_criterions()
# Search through dataset is independent
wrapper.models_manager.get_available_datasets()


 Registered Component Categories for 'autoencoder'

[Category]: 'decoder'
 Description: Component category for model family 'autoencoder'.
 Parameters (kwargs):
   - None

[Category]: 'encoder'
 Description: Component category for model family 'autoencoder'.
 Parameters (kwargs):
   - None

[Category]: 'projector'
 Description: Component category for model family 'autoencoder'.
 Parameters (kwargs):
   - None


 Available Configuration Presets for 'autoencoder'

[Preset]: 'GradualReduction'
 Description: Dynamically suggests compatible neural structures based on raw peak widths metrics.

Estimates peak envelope widths to configure matching initial convolutional fields,
iteratively scaling hidden depths until layers dimensions compress to fit bottlenecks.

:param latent_dim: Core dimension sizing assigned to the target bottleneck space.
:type latent_dim: int
:param user_hyperparameters: Manual overrides configuration maps to bypass automated heuristics.
:type user_hyperparameters: Opti

## Use a preset, then override deliberately

`GradualReduction` derives input width from the active binner and estimates a convolution kernel from sampled peak envelopes. Parameters such as latent and projection dimensions remain explicit. The preset only fills the building buffer, so individual components can still be replaced before compilation. Registered names are preferred over classes or instances because names and parameters are portable JSON.

In [9]:
wrapper.models_manager.set_model_preset(
    "GradualReduction",
    latent_dim=16, # Suggested dimension is much higher, but in this case we are using 16 for faster evaluation
    projection_dim=64,
)
wrapper.models_manager.set_dataset("PixelDataset")
model = wrapper.models_manager.compile_model(run_validation_pass=True)
print(model)

2026-07-19 13:39:07,796 | INFO     | msi_autoencoder_wrapper.core.mixins.active_context.active_context_mixin:106 | Successfully bound active context memory maps for: example
2026-07-19 13:39:07,798 | INFO     | msi_autoencoder_wrapper.core.mixins.models_manager.proxies.architecture_proxy:278 | Initiating model preset configuration layout lookup for family: autoencoder, preset: GradualReduction
2026-07-19 13:39:07,799 | INFO     | msi_autoencoder_wrapper.models.architectures.types.autoencoders.presets.gradual_reduction_preset:46 | Preset builder initiating hyperparameter execution pipeline analysis.
2026-07-19 13:39:17,765 | INFO     | msi_autoencoder_wrapper.models.architectures.utils.presets_utils:88 | Statistical reflection completed. Suggested baseline kernel width: 9 bins
2026-07-19 13:39:17,766 | INFO     | msi_autoencoder_wrapper.models.architectures.types.autoencoders.presets.gradual_reduction_preset:86 | Gradual Reduction layout synthesis finalized. Mapped feature width roadmap

For complete manual construction, call `get_available_components(category)`, then `set_component(category, registered_name, **parameters)` for encoder, decoder, and optional projector/head. Custom components and presets are documented in [Custom models](../../../docs/CUSTOM_MODELS.md).

## Define training phases

Criteria are grouped by where they act. Reconstruction losses consume input and reconstruction; contrastive losses prepare augmented inputs and consume projection outputs; head losses are reserved for named head outputs. A phase may freeze direct child modules by name. Current lifecycle hooks are criterion hooks: phase-start precomputation and batch-start augmentation. No additional inter-layer training hook API is implied here.

In [13]:
training_config = {
    "seed": 1912, # Those who know, know 
    "phases": [
        {
            "phase_name": "joint_reconstruction_and_contrastive",
            "epochs": 20,
            "batch_size": 64,
            "freeze": [],
            "optimizer": {
                "type": "AdamW",
                "params": {"lr": 1e-3, "weight_decay": 1e-4},
            },
            "criterions": {
                "reconstruction": {
                    "mse": {
                        "target": "MSELoss",
                        "weight": 1.0,
                        "params": {},
                    },
                },
                "contrastive": {
                    "info_nce": {
                        "target": "InfoNCELoss",
                        "weight": 0.05,
                        "params": {
                            "temperature": 0.07,
                            "peak_sample_size": 512,
                            "peak_sample_seed": 1912,
                        },
                    },
                },
            },
        }
    ],
}

`InfoNCELoss` samples spectra and peak envelopes once per phase, stores a bounded bank in the loaded model's transient training cache, injects sampled envelopes at batch start, and expands an `N` batch to `2N`. Clear the cache explicitly after changing data assumptions with `clear_training_cache()`. Detailed criterion contracts and the Masserstein loss are in [CRITERIONS.md](../../../docs/CRITERIONS.md).

## Estimate capacity before training

The estimator runs one evaluation forward probe and combines observed activation sizes with parameters, gradients, optimizer state, DataLoader buffering, known criterion workspaces, checkpoints, and history. Fractions in `(0, 1]` mean a fraction of currently available resources; larger values mean absolute bytes. It reports RAM, VRAM, and disk separately and can reduce batch size in a copied config. It is an estimate, not a guarantee: native reader caches, allocator fragmentation, OS activity, and future peaks are not fully measurable before training.

In [14]:
#TODO - trzeba deafult printa tutaj zrobić, tak jak przy configurajci, że automatycznei to robi str i może tobie dicty zwrócić, ale może również zrobić tak, że na returnie ustawić print'a
#
report = wrapper.models_manager.estimate_training_resources(
    training_config,
    resource_limits={"ram": 0.65, "vram": 0.80, "disk": 1.00},
    auto_adjust_batch_size=True,
    safety_factor=1.25,
)


for phase in report["phases"]:
    print(
        phase["phase"],
        phase["recommended_batch_size"],
        phase["estimated_ram_bytes"],
        phase["estimated_vram_bytes"],
        phase["fits_limits"],
    )
print(report["estimated_disk_bytes"], report["disk_fits_limit"])
safe_training_config = report["recommended_training_config"]

2026-07-19 13:49:25,262 | INFO     | msi_autoencoder_wrapper.training.resource_estimator:147 | Training resource estimation completed for 1 phase(s).
joint_reconstruction_and_contrastive 64 18694400 1507631140 True
538580 True


Masserstein allocates matrices quadratic in the number of m/z bins; InfoNCE allocates similarity matrices quadratic in the expanded batch size. 

For either loss, keep the safety reserve and monitor the first epoch on the target machine.

In [ ]:
# Training may be expensive; run after reviewing the resource report.
#TODO - there is problem with not implemented on batch start - each criterion should have default implemention of those, which be deafult do nothing, just ensure compatibility
history = wrapper.models_manager.fit(safe_training_config)
model_dir = wrapper.workspace.save_model(model_name="tutorial-autoencoder")

2026-07-19 13:49:26,091 | INFO     | msi_autoencoder_wrapper.core.mixins.models_manager.proxies.training_proxy:133 | Instantiating execution training manager instance.
2026-07-19 13:49:26,092 | INFO     | msi_autoencoder_wrapper.training.training_manager:78 | Training lifecycle orchestration triggered. Dispatching execution parameters for model family: autoencoder
2026-07-19 13:49:26,093 | INFO     | msi_autoencoder_wrapper.training.engine.base_trainer:304 | Pre-flight validation successful. Training environment maps verified.
2026-07-19 13:49:26,094 | INFO     | msi_autoencoder_wrapper.training.engine.base_trainer:63 | Enforcing global deterministic execution pipeline using seed token: 1912
2026-07-19 13:49:26,097 | INFO     | msi_autoencoder_wrapper.training.engine.base_trainer:89 | Initiating sequential training loop phase: joint_reconstruction_and_contrastive (1/1)
2026-07-19 13:49:26,098 | INFO     | msi_autoencoder_wrapper.training.criterions.criterions_manager:250 | Assembling '

/home/maxi7524/micromamba/envs/ims_env/lib/python3.12/site-packages/scipy/stats/_binned_statistic.py:376: RuntimeWarning: invalid value encountered in cast
  z = np.bincount(x, weights)
/home/maxi7524/micromamba/envs/ims_env/lib/python3.12/site-packages/scipy/stats/_binned_statistic.py:376: RuntimeWarning: invalid value encountered in cast
  z = np.bincount(x, weights)
/home/maxi7524/micromamba/envs/ims_env/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:54: RuntimeWarning: overflow encountered in multiply
  return bound(*args, **kwds)
/home/maxi7524/micromamba/envs/ims_env/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:54: RuntimeWarning: overflow encountered in multiply
  return bound(*args, **kwds)


AttributeError: 'MSIMSELoss' object has no attribute 'on_batch_start'